## F260517 Registration — Forward-Only Pipeline (First-5-Frames Warmup)

zRatio = 10, frame_rate = 4.07066 s/frame, z_window = 4.0

In [1]:
import cupy as cp
cp.cuda.Device(1).use()
import os
os.chdir('/home/cyf/wbi/Virginia/code/wbi_0123/wholistic_registration/src/wholistic_registration')
from utils import IO
from utils import calFlowCrossResolution, calFlow3d_Wei_v1
from utils import registration
from utils import option
from utils import preprocess as prep
from utils import mask
from utils import visualization
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/cyf/wbi/Virginia/code/CoarseFlow").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from training.inference import (
    CoarseFlowInferenceConfig,
    CoarseFlowPredictor,
)

F260517_mov_path = "/home/cyf/wbi/Virginia/raw_data/f260517/260517_exp_00001_TZCYX.ome.tiff"
F260517_ref_path = "/home/cyf/wbi/Virginia/raw_data/f260517/260517_anat_00003_TZCYX.ome.tiff"

import numpy as np
import pandas as pd
from scipy.ndimage import distance_transform_edt, map_coordinates, maximum_filter
from importlib import reload
from typing import Any
from collections import deque
from numpy import ndarray

print("Imports done.")

Imports done.


In [2]:
# =========================================================
# Helper functions (single copy, no duplicates)
# =========================================================

def update_reference_intensity_mapping_from_target_stack(
    F260517_ref_mem,
    target_stack_zyx,
    z_idx,
    option,
    thresFactor,
    maskRange,
    smoothPenalty_raw,
    percentiles,
):
    """
    Learn raw reference -> target intensity mapping using only z_idx planes,
    then apply the learned mapping to the full reference volume.

    F260517_ref_mem:
        Raw reference membrane volume, shape (Zref, Y, X).

    target_stack_zyx:
        Target stack used for learning intensity mapping, shape (K, Y, X).

        Initial update:
            target_stack_zyx = raw moving frame 0, F260517_mov_mem[0].

        Later updates:
            target_stack_zyx = previous/recent registered membrane image,
            i.e. mem_mapped.transpose(2, 1, 0).

    z_idx:
        Reference z planes corresponding to moving slices, shape (K,).

    Returns
    -------
    F260517_ref_mem_adj:
        Adjusted full reference volume, shape (X, Y, Zref).

    src_q, tgt_q, used_percentiles:
        Quantile mapping information.
    """

    target_stack_zyx = np.asarray(target_stack_zyx, dtype=np.float32)

    if target_stack_zyx.ndim != 3:
        raise ValueError(
            f"target_stack_zyx should be (K,Y,X), got {target_stack_zyx.shape}"
        )

    if target_stack_zyx.shape[0] != len(z_idx):
        raise ValueError(
            f"target_stack_zyx K={target_stack_zyx.shape[0]} does not match "
            f"z_idx length={len(z_idx)}"
        )

    # ------------------------------------------------------------
    # 1. Learn mapping only on matched planes.
    # ------------------------------------------------------------
    ref_source_zyx = F260517_ref_mem[z_idx].astype(np.float32, copy=False)

    if ref_source_zyx.shape != target_stack_zyx.shape:
        raise ValueError(
            f"ref_source_zyx shape {ref_source_zyx.shape} does not match "
            f"target_stack_zyx shape {target_stack_zyx.shape}"
        )

    src_q, tgt_q, used_percentiles = prep.learn_quantile_mapping(
        source=ref_source_zyx,
        target=target_stack_zyx,
        percentiles=percentiles,
    )

    # ------------------------------------------------------------
    # 2. Apply learned mapping to the full reference volume.
    # ------------------------------------------------------------
    F260517_ref_mem_adj = prep.apply_quantile_mapping(
        F260517_ref_mem,
        src_q,
        tgt_q,
    ).transpose(2, 1, 0).astype(np.float32, copy=False)  # (X, Y, Zref)

    # ------------------------------------------------------------
    # 3. Update reference mask and smooth penalty.
    # ------------------------------------------------------------
    option["mask_ref"] = mask.getMask(F260517_ref_mem_adj, thresFactor)
    option["mask_ref"] = mask.bwareafilt3_wei(option["mask_ref"], maskRange)

    Pnltfactor = prep.getSmPnltNormFctr(F260517_ref_mem_adj, option)
    option["smoothPenalty"] = Pnltfactor * smoothPenalty_raw

    return F260517_ref_mem_adj, src_q, tgt_q, used_percentiles




def robust_average_target_z(target_z_list, method="median", trim_percentiles=(10, 90)):
    """
    Combine target_z estimates from multiple warmup frames.

    target_z_list:
        list of arrays, each shape (K,)

    return:
        fixed_target_z, shape (K,)
    """
    arr = np.stack([np.asarray(z, dtype=np.float32) for z in target_z_list], axis=0)

    if method == "median":
        fixed_z = np.nanmedian(arr, axis=0)

    elif method == "mean":
        fixed_z = np.nanmean(arr, axis=0)

    elif method == "trimmed_mean":
        lo_p, hi_p = trim_percentiles
        fixed = []

        for k_idx in range(arr.shape[1]):
            vals = arr[:, k_idx]
            vals = vals[np.isfinite(vals)]

            if vals.size == 0:
                fixed.append(np.nan)
                continue

            lo = np.percentile(vals, lo_p)
            hi = np.percentile(vals, hi_p)
            vals_trim = vals[(vals >= lo) & (vals <= hi)]

            if vals_trim.size == 0:
                fixed.append(float(np.median(vals)))
            else:
                fixed.append(float(np.mean(vals_trim)))

        fixed_z = np.asarray(fixed, dtype=np.float32)

    else:
        raise ValueError("method should be 'median', 'mean', or 'trimmed_mean'.")

    return fixed_z.astype(np.float32)


def estimate_projection_z_from_phase_simple(
    phase_new,
    z_init,
    ref_shape,
    ref_volume_order="zyx",
    method="trimmed_mean",
    trim_percentiles=(5, 95),
    frame_idx=None,
):
    """
    Estimate one target reference z plane for each moving slice k
    from phase_new[..., 2].

    phase_new:
        shape (X, Y, K, 3), xyz coordinates in reference volume

    z_init:
        shape (K,)

    ref_shape:
        if ref_volume_order == "zyx": (Zref, Yref, Xref)
        if ref_volume_order == "xyz": (Xref, Yref, Zref)
    """
    if hasattr(phase_new, "get"):
        phase = phase_new.get()
    else:
        phase = np.asarray(phase_new)

    phase = np.asarray(phase, dtype=np.float32)
    z_init = np.asarray(z_init, dtype=np.float32)

    if phase.ndim != 4 or phase.shape[-1] != 3:
        raise ValueError(f"phase_new should be (X,Y,K,3), got {phase.shape}")

    X, Y, K_local, _ = phase.shape

    if len(z_init) != K_local:
        raise ValueError(f"z_init length {len(z_init)} does not match K={K_local}")

    x_ref = phase[..., 0]
    y_ref = phase[..., 1]
    z_ref = phase[..., 2]

    if ref_volume_order == "zyx":
        Zref, Yref, Xref = ref_shape
    elif ref_volume_order == "xyz":
        Xref, Yref, Zref = ref_shape
    else:
        raise ValueError("ref_volume_order should be 'zyx' or 'xyz'.")

    valid = (
        np.isfinite(x_ref)
        & np.isfinite(y_ref)
        & np.isfinite(z_ref)
        & (x_ref >= 0) & (x_ref <= Xref - 1)
        & (y_ref >= 0) & (y_ref <= Yref - 1)
        & (z_ref >= 0) & (z_ref <= Zref - 1)
    )

    target_z_pred = np.zeros(K_local, dtype=np.float32)
    rows = []

    for k_idx in range(K_local):
        z_k = z_ref[:, :, k_idx]
        valid_k = valid[:, :, k_idx]
        z_valid = z_k[valid_k]

        if z_valid.size == 0:
            z_est = float(z_init[k_idx])

        else:
            if method == "mean":
                z_est = float(np.mean(z_valid))

            elif method == "median":
                z_est = float(np.median(z_valid))

            elif method == "trimmed_mean":
                p_low, p_high = trim_percentiles
                lo = np.percentile(z_valid, p_low)
                hi = np.percentile(z_valid, p_high)
                z_trim = z_valid[(z_valid >= lo) & (z_valid <= hi)]

                if z_trim.size == 0:
                    z_est = float(np.median(z_valid))
                else:
                    z_est = float(np.mean(z_trim))

            else:
                raise ValueError("method should be 'mean', 'median', or 'trimmed_mean'.")

        target_z_pred[k_idx] = z_est

        if z_valid.size > 0:
            dz_valid = z_valid - float(z_init[k_idx])

            rows.append({
                "frame": frame_idx,
                "k": k_idx,
                "z_init": float(z_init[k_idx]),
                "target_z_pred": float(z_est),
                "target_minus_z_init": float(z_est - z_init[k_idx]),
                "z_mean": float(np.mean(z_valid)),
                "z_median": float(np.median(z_valid)),
                "z_p05": float(np.percentile(z_valid, 5)),
                "z_p95": float(np.percentile(z_valid, 95)),
                "dz_mean": float(np.mean(dz_valid)),
                "dz_median": float(np.median(dz_valid)),
                "dz_p05": float(np.percentile(dz_valid, 5)),
                "dz_p95": float(np.percentile(dz_valid, 95)),
                "valid_ratio": float(z_valid.size / (X * Y)),
            })

        else:
            rows.append({
                "frame": frame_idx,
                "k": k_idx,
                "z_init": float(z_init[k_idx]),
                "target_z_pred": float(z_est),
                "target_minus_z_init": float(z_est - z_init[k_idx]),
                "z_mean": np.nan,
                "z_median": np.nan,
                "z_p05": np.nan,
                "z_p95": np.nan,
                "dz_mean": np.nan,
                "dz_median": np.nan,
                "dz_p05": np.nan,
                "dz_p95": np.nan,
                "valid_ratio": 0.0,
            })

    return target_z_pred, pd.DataFrame(rows)


# =========================================================
# Supersurface / surface supersampling helpers
# =========================================================


def _make_xy_sampling_grid(X, Y, upsample_factor):
    """
    Build a dense sampling grid in moving XY parameter domain.

    Original grid:
        x = 0, 1, ..., X-1
        y = 0, 1, ..., Y-1

    Upsampled grid covers the same coordinate range [0, X-1] and [0, Y-1].
    """
    factor = int(upsample_factor)

    if factor < 1:
        raise ValueError("upsample_factor should be >= 1.")

    Xup = int(X * factor)
    Yup = int(Y * factor)

    x_new = np.linspace(0, X - 1, Xup, dtype=np.float32)
    y_new = np.linspace(0, Y - 1, Yup, dtype=np.float32)

    Xg, Yg = np.meshgrid(x_new, y_new, indexing="ij")

    coords_2d = np.vstack([
        Xg.reshape(-1),
        Yg.reshape(-1),
    ])

    return coords_2d, Xup, Yup


def upsample_phase_xy_for_supersurface(
    phase_new,
    upsample_factor=2,
):
    """
    Upsample phase_new in moving XY parameter domain.

    phase_new:
        shape (X, Y, K, 3)

    output:
        shape (X*factor, Y*factor, K, 3)

    This is a practical approximation to sampling a continuous moving surface.
    """
    phase = np.asarray(phase_new, dtype=np.float32)

    if phase.ndim != 4 or phase.shape[-1] != 3:
        raise ValueError(f"phase_new should be (X,Y,K,3), got {phase.shape}")

    factor = int(upsample_factor)

    if factor == 1:
        return phase

    X, Y, K_local, C = phase.shape
    coords_2d, Xup, Yup = _make_xy_sampling_grid(X, Y, factor)

    phase_up = np.empty((Xup, Yup, K_local, C), dtype=np.float32)

    for k_idx in range(K_local):
        for c_idx in range(C):
            phase_up[:, :, k_idx, c_idx] = map_coordinates(
                phase[:, :, k_idx, c_idx],
                coords_2d,
                order=1,
                mode="nearest",
            ).reshape(Xup, Yup)

    return phase_up




def upsample_values_xy_for_supersurface(
    values_xyk,
    upsample_factor=2,
    order=1,
):
    """
    Upsample values in moving XY parameter domain.

    values_xyk:
        shape (X, Y, K)

    order:
        1: bilinear-like interpolation, suitable for membrane.
        0: nearest interpolation, suitable for sparse bright signal.
    """
    values = np.asarray(values_xyk, dtype=np.float32)

    if values.ndim != 3:
        raise ValueError(f"values_xyk should be (X,Y,K), got {values.shape}")

    factor = int(upsample_factor)

    if factor == 1:
        return values

    X, Y, K_local = values.shape
    coords_2d, Xup, Yup = _make_xy_sampling_grid(X, Y, factor)

    values_up = np.empty((Xup, Yup, K_local), dtype=np.float32)

    for k_idx in range(K_local):
        values_up[:, :, k_idx] = map_coordinates(
            values[:, :, k_idx],
            coords_2d,
            order=order,
            mode="nearest",
        ).reshape(Xup, Yup)

    return values_up


# =========================================================
# Output / diagnostics helpers
# =========================================================


def save_single_channel_ome_tiff(
    volume_zyx,
    out_dir,
    frame_idx,
    label="F260517",
):
    """
    Save one volume as a single-channel OME-TIFF in one output folder.

    volume_zyx:
        shape (Z, Y, X)
    """
    os.makedirs(out_dir, exist_ok=True)

    IO.write_multichannel_volume_as_ome_tiff(
        volume=[np.asarray(volume_zyx, dtype=np.float32)],
        out_dir=out_dir,
        frame_idx=frame_idx,
        label=label,
    )




def summarize_coverage_only(
    frame_idx,
    coverage,
    coverage_threshold=0.0,
):
    """
    Summarize coverage map statistics.

    coverage:
        shape (Nplanes, Y, X)

    coverage <= threshold is treated as no coverage.
    """
    coverage = np.asarray(coverage, dtype=np.float32)

    if coverage.ndim != 3:
        raise ValueError(f"coverage should be (Nplanes,Y,X), got {coverage.shape}")

    no_coverage = coverage <= coverage_threshold

    no_cov_per_plane = np.mean(no_coverage, axis=(1, 2))
    cov_mean_per_plane = np.mean(coverage, axis=(1, 2))
    cov_max_per_plane = np.max(coverage, axis=(1, 2))

    global_no_cov = float(np.mean(no_coverage))
    worst_k = int(np.argmax(no_cov_per_plane))

    print(
        f"[Coverage] frame {frame_idx}: "
        f"global_no_cov={global_no_cov:.4f}, "
        f"min_plane_no_cov={float(no_cov_per_plane.min()):.4f}, "
        f"median_plane_no_cov={float(np.median(no_cov_per_plane)):.4f}, "
        f"max_plane_no_cov={float(no_cov_per_plane.max()):.4f}, "
        f"worst_k={worst_k}"
    )

    rows = []

    rows.append({
        "frame": int(frame_idx),
        "plane": -1,
        "is_global": True,
        "no_coverage_ratio": global_no_cov,
        "coverage_ratio": 1.0 - global_no_cov,
        "coverage_mean": float(np.mean(coverage)),
        "coverage_max": float(np.max(coverage)),
        "worst_plane_by_no_coverage": worst_k,
        "max_plane_no_coverage_ratio": float(no_cov_per_plane.max()),
    })

    for k_idx in range(coverage.shape[0]):
        rows.append({
            "frame": int(frame_idx),
            "plane": int(k_idx),
            "is_global": False,
            "no_coverage_ratio": float(no_cov_per_plane[k_idx]),
            "coverage_ratio": float(1.0 - no_cov_per_plane[k_idx]),
            "coverage_mean": float(cov_mean_per_plane[k_idx]),
            "coverage_max": float(cov_max_per_plane[k_idx]),
            "worst_plane_by_no_coverage": worst_k,
            "max_plane_no_coverage_ratio": float(no_cov_per_plane.max()),
        })

    return rows




def fill_holes_nearest_limited(
    proj_zyx,
    fill_value=-200.0,
    max_fill_distance=2.0,
    hole_tol=1e-6,
):
    """
    Fill only small holes using nearest valid pixels.

    This function does NOT smooth the whole image.
    It only modifies pixels equal to fill_value.

    proj_zyx:
        shape (Z, Y, X)

    max_fill_distance:
        Only holes within this distance from valid pixels are filled.
    """
    proj = np.asarray(proj_zyx, dtype=np.float32)
    out = proj.copy()

    for zi in range(out.shape[0]):
        plane = out[zi]

        hole = np.abs(plane - fill_value) <= hole_tol
        valid = (~hole) & np.isfinite(plane)

        if not np.any(hole) or not np.any(valid):
            continue

        dist, indices = distance_transform_edt(
            ~valid,
            return_indices=True,
        )

        fill_mask = hole & (dist <= max_fill_distance)

        nearest_y = indices[0][fill_mask]
        nearest_x = indices[1][fill_mask]

        plane[fill_mask] = plane[nearest_y, nearest_x]
        out[zi] = plane

    return out




def project_converge_map_gpu(
    phase_new,
    ref_volume_for_shape,
    fixed_target_z,
    ref_volume_order="xyz",
    z_window=3.0,
    downsample_xy=1,
    xy_extra_radius=0,
):
    """
    Return coverage map for current phase_new.

    Output:
        coverage_zyx, shape (Nplanes, Y, X)
        value > 0 means at least one sample hit this pixel.
    """
    values_ones = np.ones(phase_new.shape[:-1], dtype=np.float32)

    coverage = calFlowCrossResolution.project_coords_to_fixed_planes_gpu(
        coords_ref_xyk_xyz=phase_new,
        ref_volume=ref_volume_for_shape,
        target_z_planes=fixed_target_z,
        values_xyk=values_ones,
        ref_volume_order=ref_volume_order,
        z_window=z_window,
        downsample_xy=downsample_xy,
        fill_value=0.0,
        return_numpy=True,
        output_order="zyx",
        xy_splat_mode="subpixel_footprint",
        xy_extra_radius=xy_extra_radius,
    )

    return coverage



def project_and_save_single_frame_moving_derived(
    frame_idx,
    phase_new,
    fixed_target_z,
    ref_mem_adj_for_projection,
    F260517_ref_sparseCell,
    raw_mem_zyx,
    raw_sparseCell_zyx,
    out_dirs,
    surface_upsample_factor=2,
    surface_mem_value_order=1,
    surface_sparse_value_order=0,
    enable_coverage_diagnostics=True,
    coverage_threshold=0.0,
    enable_hole_filling=False,
):
    """
    Save four outputs into four separate folders:

        1. raw moving membrane
        2. raw moving sparse-cell
        3. projected moving membrane
        4. projected moving sparse-cell

    Projected values come from current raw moving images, not reference:

        values_xyk = raw_mem_zyx.transpose(2, 1, 0)
        values_xyk = raw_sparseCell_zyx.transpose(2, 1, 0)

    Supersurface:
        If surface_upsample_factor > 1, phase_new and moving values are
        upsampled in moving XY parameter domain before projection.
    """
    phase_new = np.asarray(phase_new, dtype=np.float32)

    raw_mem_zyx = np.asarray(raw_mem_zyx, dtype=np.float32)
    raw_sparseCell_zyx = np.asarray(raw_sparseCell_zyx, dtype=np.float32)

    raw_mem_xyk = raw_mem_zyx.transpose(2, 1, 0).astype(np.float32, copy=False)
    raw_sparse_xyk = raw_sparseCell_zyx.transpose(2, 1, 0).astype(np.float32, copy=False)

    # ------------------------------------------------------------
    # 1. Save raw moving channels
    # ------------------------------------------------------------
    save_single_channel_ome_tiff(
        volume_zyx=raw_mem_zyx,
        out_dir=out_dirs["raw_moving_mem"],
        frame_idx=frame_idx,
        label="F260517_raw_mem",
    )

    save_single_channel_ome_tiff(
        volume_zyx=raw_sparseCell_zyx,
        out_dir=out_dirs["raw_moving_sparseCell"],
        frame_idx=frame_idx,
        label="F260517_raw_sparseCell",
    )

    # ------------------------------------------------------------
    # 2. Supersurface sampling
    # ------------------------------------------------------------
    if int(surface_upsample_factor) > 1:
        print(
            f"[Supersurface] frame {frame_idx}: "
            f"upsample_factor={surface_upsample_factor}"
        )

    phase_for_projection = upsample_phase_xy_for_supersurface(
        phase_new,
        upsample_factor=surface_upsample_factor,
    )

    mem_values_for_projection = upsample_values_xy_for_supersurface(
        raw_mem_xyk,
        upsample_factor=surface_upsample_factor,
        order=surface_mem_value_order,
    )

    sparse_values_for_projection = upsample_values_xy_for_supersurface(
        raw_sparse_xyk,
        upsample_factor=surface_upsample_factor,
        order=surface_sparse_value_order,
    )

    if phase_for_projection.shape[:-1] != mem_values_for_projection.shape:
        raise ValueError(
            f"phase_for_projection shape {phase_for_projection.shape[:-1]} "
            f"does not match mem_values_for_projection shape {mem_values_for_projection.shape}"
        )

    if phase_for_projection.shape[:-1] != sparse_values_for_projection.shape:
        raise ValueError(
            f"phase_for_projection shape {phase_for_projection.shape[:-1]} "
            f"does not match sparse_values_for_projection shape {sparse_values_for_projection.shape}"
        )

    # ------------------------------------------------------------
    # 3. Project raw moving membrane to fixed reference planes
    # ------------------------------------------------------------
    mem_proj = calFlowCrossResolution.project_coords_to_fixed_planes_gpu(
        coords_ref_xyk_xyz=phase_for_projection,
        ref_volume=ref_mem_adj_for_projection,       # used as reference canvas
        target_z_planes=fixed_target_z,
        values_xyk=mem_values_for_projection,        # values come from moving
        ref_volume_order="xyz",
        z_window=projection_z_window,
        downsample_xy=projection_downsample_xy,
        fill_value=projection_fill_value,
        return_numpy=True,
        output_order="zyx",
        xy_splat_mode="subpixel_footprint",
        xy_extra_radius=projection_xy_extra_radius,
    )

    # ------------------------------------------------------------
    # 4. Project raw moving sparse-cell to fixed reference planes
    # ------------------------------------------------------------
    sparse_proj = calFlowCrossResolution.project_coords_to_fixed_planes_gpu(
        coords_ref_xyk_xyz=phase_for_projection,
        ref_volume=F260517_ref_sparseCell,            # used as reference canvas
        target_z_planes=fixed_target_z,
        values_xyk=sparse_values_for_projection,      # values come from moving
        ref_volume_order="zyx",
        z_window=projection_z_window,
        downsample_xy=projection_downsample_xy,
        fill_value=projection_fill_value,
        return_numpy=True,
        output_order="zyx",
        xy_splat_mode="subpixel_footprint",
        xy_extra_radius=projection_xy_extra_radius,
    )

    coverage_rows = []

    # ------------------------------------------------------------
    # 5. Coverage diagnostics only
    # ------------------------------------------------------------
    if enable_coverage_diagnostics:
        coverage = project_converge_map_gpu(
            phase_new=phase_for_projection,
            ref_volume_for_shape=ref_mem_adj_for_projection,
            fixed_target_z=fixed_target_z,
            ref_volume_order="xyz",
            z_window=projection_z_window,
            downsample_xy=projection_downsample_xy,
            xy_extra_radius=projection_xy_extra_radius,
        )

        if hasattr(coverage, "get"):
            coverage = coverage.get()

        coverage = np.asarray(coverage, dtype=np.float32)

        if coverage.shape != mem_proj.shape:
            raise ValueError(
                f"coverage shape {coverage.shape} does not match projection shape {mem_proj.shape}. "
                "Please check project_converge_map_gpu output_order / downsample settings."
            )

        coverage_rows = summarize_coverage_only(
            frame_idx=frame_idx,
            coverage=coverage,
            coverage_threshold=coverage_threshold,
        )

    # ------------------------------------------------------------
    # 6. Optional hole filling from moving-derived projection only
    # ------------------------------------------------------------
    if enable_hole_filling:
        mem_proj = fill_holes_nearest_limited(
            mem_proj,
            fill_value=projection_fill_value,
            max_fill_distance=2.0,
        )

        sparse_proj = fill_holes_nearest_limited(
            sparse_proj,
            fill_value=projection_fill_value,
            max_fill_distance=1.5,
        )

    # ------------------------------------------------------------
    # 7. Save projected channels
    # ------------------------------------------------------------
    save_single_channel_ome_tiff(
        volume_zyx=mem_proj,
        out_dir=out_dirs["projected_mem"],
        frame_idx=frame_idx,
        label="F260517_projected_mem_from_moving",
    )

    save_single_channel_ome_tiff(
        volume_zyx=sparse_proj,
        out_dir=out_dirs["projected_sparseCell"],
        frame_idx=frame_idx,
        label="F260517_projected_sparseCell_from_moving",
    )

    del (
        mem_proj,
        sparse_proj,
        phase_for_projection,
        mem_values_for_projection,
        sparse_values_for_projection,
    )

    if "cleanup_gpu_memory" in globals():
        cleanup_gpu_memory()

    return coverage_rows


print("Helper functions defined.")

Helper functions defined.


In [3]:
# =========================================================
# Data loading
# =========================================================
reload(calFlowCrossResolution)
reload(prep)

F260517_mov, F260517_mov_desc = IO.readTiff(F260517_mov_path)
F260517_ref, F260517_ref_desc = IO.readTiff(F260517_ref_path)
print("Finish reading the dataset")

# =========================================================
# Split into 2 stacks
# =========================================================
F260517_ref_mem = F260517_ref[90:310, 1, :, :]
F260517_ref_sparseCell = F260517_ref[90:310, 0, :, :]

F260517_mov_mem: ndarray[tuple[int, ...], Any] = F260517_mov[:, :, 1, :, :]
F260517_mov_sparseCell: ndarray[tuple[int, ...], Any] = F260517_mov[:, :, 0, :, :]

print("F260517_ref_mem:", F260517_ref_mem.shape)          # (Zref, Y, X)
print("F260517_ref_sparseCell:", F260517_ref_sparseCell.shape)
print("F260517_mov_mem:", F260517_mov_mem.shape)          # (T, K, Y, X)
print("F260517_mov_sparseCell:", F260517_mov_sparseCell.shape)
print("Finish splitting the dataset")

# =========================================================
# Registration options
# =========================================================
option["r"] = 5
option["layer"] = 3
option["iter"] = 10
option["movRange"] = 5.0
option["tol"] = 1e-6
option["zRatio_HR"] = 1
option["wrong_region_enable"] = False

thresFactor = 5.0
maskRange = [5.0, 4000.0]
smoothPenalty_raw = 0.01

# =========================================================
# Find initial corresponding z slices
# =========================================================
z_init = calFlowCrossResolution.FindInitZ_stack_global_fixed_spacing(
    F260517_mov_mem[0].transpose(2, 1, 0),     # (X, Y, K)
    F260517_ref_mem.transpose(2, 1, 0),        # (X, Y, Zref)
    delta_ref_idx=10,
    use_gradient=False,
)

z_init = np.asarray(z_init, dtype=np.float32)
z_idx = np.rint(z_init).astype(np.int32)
z_idx = np.clip(z_idx, 0, F260517_ref_mem.shape[0] - 1)

K = z_init.shape[0]
T = F260517_mov_mem.shape[0]

x = np.arange(F260517_mov_mem[0].shape[2], dtype=np.float32)
y = np.arange(F260517_mov_mem[0].shape[1], dtype=np.float32)
k = np.arange(K, dtype=np.int32)

X_grid, Y_grid, K_grid = np.meshgrid(x, y, k, indexing="ij")

coords_xyz = np.empty(
    (F260517_mov_mem[0].shape[2], F260517_mov_mem[0].shape[1], K, 3),
    dtype=np.float32,
)
coords_xyz[..., 0] = X_grid
coords_xyz[..., 1] = Y_grid
coords_xyz[..., 2] = z_init[K_grid]

option["phase"] = coords_xyz

print("z_init:", z_init)
print("z_idx:", z_idx)
print("phase shape:", option["phase"].shape)
print("T (total frames):", T)

# =========================================================
# Quantile mapping percentiles
# =========================================================
percentiles = [
    0.1, 0.5, 1, 2, 5,
    10, 25, 50, 75, 90,
    95, 99, 99.5, 99.8,
]

# =========================================================
# Projection settings
# =========================================================
projection_z_window = 3.0
projection_downsample_xy = 1
projection_fill_value = -200.0
projection_xy_extra_radius = 0

surface_upsample_factor = 2
surface_mem_value_order = 1
surface_sparse_value_order = 0

enable_coverage_diagnostics = True
coverage_threshold = 0.0
enable_projection_hole_filling = False

# =========================================================
# Output directories
# =========================================================
base_out_dir = "/home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0609/"

out_dirs = {
    "raw_moving_mem": os.path.join(base_out_dir, "raw_moving_mem"),
    "raw_moving_sparseCell": os.path.join(base_out_dir, "raw_moving_sparseCell"),
    "projected_mem": os.path.join(base_out_dir, "projected_mem"),
    "projected_sparseCell": os.path.join(base_out_dir, "projected_sparseCell"),
}

for _d in out_dirs.values():
    os.makedirs(_d, exist_ok=True)

diagnostic_dir = os.path.join(base_out_dir, "diagnostics")
os.makedirs(diagnostic_dir, exist_ok=True)

# =========================================================
# Reference update settings
# =========================================================
ref_update_every = 40

print("Initial setup done.")

Finish reading the dataset
F260517_ref_mem: (220, 1500, 630)
F260517_ref_sparseCell: (220, 1500, 630)
F260517_mov_mem: (200, 20, 1500, 630)
F260517_mov_sparseCell: (200, 20, 1500, 630)
Finish splitting the dataset
z_init: [ 20.  30.  40.  50.  60.  70.  80.  90. 100. 110. 120. 130. 140. 150.
 160. 170. 180. 190. 200. 210.]
z_idx: [ 20  30  40  50  60  70  80  90 100 110 120 130 140 150 160 170 180 190
 200 210]
phase shape: (630, 1500, 20, 3)
T (total frames): 200
Initial setup done.


In [4]:
# =========================================================
# Initial reference intensity mapping
# Use mean of raw frames 0,1,2,3,4 (beginning of video)
# =========================================================
init_calibration_frames = [0, 1, 2, 3, 4]
init_target_stack = np.mean(
    F260517_mov_mem[init_calibration_frames].astype(np.float32),
    axis=0,
)  # (K, Y, X)

print("Initial calibration frames:", init_calibration_frames)
print("Target stack shape:", init_target_stack.shape)

F260517_ref_mem_adj, src_q, tgt_q, used_percentiles = (
    update_reference_intensity_mapping_from_target_stack(
        F260517_ref_mem=F260517_ref_mem,
        target_stack_zyx=init_target_stack,
        z_idx=z_idx,
        option=option,
        thresFactor=thresFactor,
        maskRange=maskRange,
        smoothPenalty_raw=smoothPenalty_raw,
        percentiles=percentiles,
    )
)

print("[Init] Updated reference intensity mapping using mean of raw frames", init_calibration_frames)
print("src_q:", src_q)
print("tgt_q:", tgt_q)

Initial calibration frames: [0, 1, 2, 3, 4]
Target stack shape: (20, 1500, 630)
[Init] Updated reference intensity mapping using mean of raw frames [0, 1, 2, 3, 4]
src_q: [ -188.     -156.     -140.     -120.      -87.      -53.       26.
   314.     1897.     4546.     5872.     8854.    10194.    11971.002]
tgt_q: [-118.4    -102.6     -94.6     -85.6     -71.      -56.4     -22.4
  151.     1086.8    2605.2    3329.6    4888.2    5581.8    6505.8003]


In [5]:
# =========================================================
# Forward-only registration pipeline
#
#   1. Warmup: register frames 0→4, determine fixed_target_z
#   2. Forward: process 5→T, update ref every 40 frames
#      - Each ref update uses the most recent 5 registered frames
# =========================================================

# ------------------------------------------------------------
# Global state
# ------------------------------------------------------------
registered_mem_mapped = {}   # frame_idx -> mem_mapped_zyx (K, Y, X)
registered_motion = {}        # frame_idx -> motion_current (X, Y, K, 3)
processed_count = 0
frames_since_ref_update = 0

target_z_pred_list = []
all_z_plane_stats = []
all_coverage_stats = []
records_warmup = []

fixed_target_z = None
target_z_combine_method = "median"
warmup_n = 5

# ------------------------------------------------------------
# Helper: update reference from recent registered frames
# ------------------------------------------------------------
def update_ref_from_recent_frames(frame_indices):
    """Calibrate reference intensity using the mean of the RAW MOVING frames at
    the given indices (consistent with the initial calibration). The raw moving
    data is what actually carries intensity drift over time; the warped-reference
    reconstruction in registered_mem_mapped does not, so it must not be the
    calibration target."""
    stacks = []
    for fi in frame_indices:
        if fi in registered_mem_mapped:
            stacks.append(F260517_mov_mem[fi].astype(np.float32, copy=False))
        else:
            print(f"  WARNING: frame {fi} not processed yet, skipping for ref update")

    if len(stacks) == 0:
        print("  WARNING: no cached frames available for ref update, skipping")
        return None, None, None

    target_stack = np.mean(np.stack(stacks, axis=0), axis=0).astype(np.float32, copy=False)

    ref_adj, sq, tq, up = update_reference_intensity_mapping_from_target_stack(
        F260517_ref_mem=F260517_ref_mem,
        target_stack_zyx=target_stack,
        z_idx=z_idx,
        option=option,
        thresFactor=thresFactor,
        maskRange=maskRange,
        smoothPenalty_raw=smoothPenalty_raw,
        percentiles=percentiles,
    )

    print(f"[RefUpdate] Calibrated from frames {frame_indices}, using {len(stacks)} stacks")
    print(f"  src_q: {sq}")
    print(f"  tgt_q: {tq}")

    return ref_adj, sq, tq

# ------------------------------------------------------------
# Helper: process a single frame
# ------------------------------------------------------------
def process_single_frame(i, ref_mem_adj):
    """Register one frame. Returns a dict with results."""
    raw_mem_zyx = F260517_mov_mem[i].astype(np.float32, copy=False)
    raw_sparse_zyx = F260517_mov_sparseCell[i].astype(np.float32, copy=False)

    mov_mem_i = raw_mem_zyx.transpose(2, 1, 0).astype(np.float32, copy=False)  # (X, Y, K)

    # Moving mask
    option["mask_mov"] = mask.getMask(mov_mem_i, thresFactor)
    option["mask_mov"] = mask.bwareafilt3_wei(option["mask_mov"], maskRange)

    # Registration
    phase_new, motion_current, mem_mapped = calFlowCrossResolution.getMotion_v2(
        mov_mem_i, ref_mem_adj, option, verbose=False,
    )

    # Convert from cupy if needed
    if hasattr(phase_new, "get"):
        phase_new = phase_new.get()
    if hasattr(motion_current, "get"):
        motion_current = motion_current.get()
    if hasattr(mem_mapped, "get"):
        mem_mapped = mem_mapped.get()

    phase_new = phase_new.astype(np.float32, copy=False)
    motion_current = motion_current.astype(np.float32, copy=False)
    mem_mapped = mem_mapped.astype(np.float32, copy=False)

    mem_err = float(np.mean(np.abs(mov_mem_i - mem_mapped)))

    # Cache mem_mapped (K, Y, X) for ref updates
    mem_mapped_zyx = mem_mapped.transpose(2, 1, 0).astype(np.float32, copy=False)

    # Estimate target z
    target_z_pred_i, df_z_i = estimate_projection_z_from_phase_simple(
        phase_new=phase_new,
        z_init=z_init,
        ref_shape=F260517_ref_mem.shape,
        ref_volume_order="zyx",
        method="trimmed_mean",
        trim_percentiles=(5, 95),
        frame_idx=i,
    )

    # Temporal initialization for next frame
    option["motion"] = (0.7 * motion_current).astype(np.float32, copy=False)

    return {
        "frame": i,
        "phase_new": phase_new,
        "motion_current": motion_current,
        "mem_mapped_zyx": mem_mapped_zyx,
        "mem_err": mem_err,
        "target_z_pred": target_z_pred_i,
        "df_z": df_z_i,
        "raw_mem_zyx": raw_mem_zyx,
        "raw_sparse_zyx": raw_sparse_zyx,
    }

# ------------------------------------------------------------
# Helper: project and save a frame result
# ------------------------------------------------------------
def project_and_save_result(result, ref_mem_adj_to_use):
    """Project a single frame result to fixed planes and save."""
    coverage_rows = project_and_save_single_frame_moving_derived(
        frame_idx=result["frame"],
        phase_new=result["phase_new"],
        fixed_target_z=fixed_target_z,
        ref_mem_adj_for_projection=ref_mem_adj_to_use,
        F260517_ref_sparseCell=F260517_ref_sparseCell,
        raw_mem_zyx=result["raw_mem_zyx"],
        raw_sparseCell_zyx=result["raw_sparse_zyx"],
        out_dirs=out_dirs,
        surface_upsample_factor=surface_upsample_factor,
        surface_mem_value_order=surface_mem_value_order,
        surface_sparse_value_order=surface_sparse_value_order,
        enable_coverage_diagnostics=enable_coverage_diagnostics,
        coverage_threshold=coverage_threshold,
        enable_hole_filling=enable_projection_hole_filling,
    )
    return coverage_rows

# ------------------------------------------------------------
# Initialize for first frame
# ------------------------------------------------------------
option["phase"] = coords_xyz.copy()
option.pop("motion", None)

# ============================================================
# Phase 1: Warmup — register first 5 frames, determine fixed_z
# ============================================================
print("=" * 80)
print("Phase 1: Warmup — registering frames 0→4")
print("=" * 80)

warmup_frames = [0, 1, 2, 3, 4]

for idx, i in enumerate(warmup_frames):
    # First warmup frame uses coords_xyz init; subsequent use temporal init
    if idx == 0:
        option["phase"] = coords_xyz.copy()
        option.pop("motion", None)

    result = process_single_frame(i, F260517_ref_mem_adj)

    registered_mem_mapped[i] = result["mem_mapped_zyx"]
    registered_motion[i] = result["motion_current"]
    target_z_pred_list.append(result["target_z_pred"])
    all_z_plane_stats.append(result["df_z"])
    records_warmup.append(result)

    processed_count += 1
    frames_since_ref_update += 1

    print(f"[Warmup {idx+1}/{warmup_n}] Frame {i}: mem_err={result['mem_err']:.4f}")

# Determine fixed_target_z from warmup frames
fixed_target_z = robust_average_target_z(
    target_z_pred_list,
    method=target_z_combine_method,
    trim_percentiles=(10, 90),
)
bad = ~np.isfinite(fixed_target_z)
fixed_target_z[bad] = z_init[bad]

print("=" * 80)
print(f"[Projection] fixed_target_z determined from {len(warmup_frames)} warmup frames")
print("z_init:", z_init)
print("fixed_target_z:", fixed_target_z)
print("fixed_target_z - z_init:", fixed_target_z - z_init)
print("=" * 80)

# Flush warmup frames — project and save
for rec in records_warmup:
    coverage_rows = project_and_save_result(rec, F260517_ref_mem_adj)
    all_coverage_stats.extend(coverage_rows)
    print(f"[Projection] saved warmup frame {rec['frame']} | mem_err={rec['mem_err']:.4f}")

records_warmup.clear()

# ============================================================
# Phase 2: Forward 5→T, with ref update every 40 frames
# ============================================================
print("=" * 80)
print("Phase 2: Forward 5→T")
print("=" * 80)

for i in range(5, T):
    if i > 75:
        option["wrong_region_enable"] = True

    result = process_single_frame(i, F260517_ref_mem_adj)

    registered_mem_mapped[i] = result["mem_mapped_zyx"]
    registered_motion[i] = result["motion_current"]
    all_z_plane_stats.append(result["df_z"])

    processed_count += 1
    frames_since_ref_update += 1

    # Project and save
    coverage_rows = project_and_save_result(result, F260517_ref_mem_adj)
    all_coverage_stats.extend(coverage_rows)

    print(f"[Fwd] Frame {i}: mem_err={result['mem_err']:.4f} | "
          f"processed={processed_count} | since_ref_update={frames_since_ref_update}")

    # Reference update every 40 frames — always use most recent 5 frames
    if frames_since_ref_update >= ref_update_every:
        calib_frames = sorted(registered_mem_mapped.keys())[-5:]
        print(f"\\n[RefUpdate] Updating reference mapping using recent frames {calib_frames}")

        new_ref, sq, tq = update_ref_from_recent_frames(calib_frames)
        if new_ref is not None:
            F260517_ref_mem_adj = new_ref
        frames_since_ref_update = 0

print("=" * 80)
print(f"Pipeline complete. Processed {processed_count} frames total.")
print(f"Frames in cache: {len(registered_mem_mapped)}")
print("=" * 80)

Phase 1: Warmup — registering frames 0→4
[Warmup 1/5] Frame 0: mem_err=170.3118
[Warmup 2/5] Frame 1: mem_err=170.6604
[Warmup 3/5] Frame 2: mem_err=171.1361
[Warmup 4/5] Frame 3: mem_err=170.6991
[Warmup 5/5] Frame 4: mem_err=170.4696
[Projection] fixed_target_z determined from 5 warmup frames
z_init: [ 20.  30.  40.  50.  60.  70.  80.  90. 100. 110. 120. 130. 140. 150.
 160. 170. 180. 190. 200. 210.]
fixed_target_z: [ 19.863358  29.803465  40.24072   49.930035  60.665195  70.726494
  81.0346    90.67564  100.340935 110.90576  121.02015  130.83269
 140.49748  150.36488  160.74434  170.84966  180.75089  190.57024
 200.55981  210.44498 ]
fixed_target_z - z_init: [-0.13664246 -0.19653511  0.24071884 -0.06996536  0.66519547  0.72649384
  1.0345993   0.6756363   0.34093475  0.9057617   1.0201492   0.8326874
  0.4974823   0.36488342  0.744339    0.84965515  0.750885    0.5702362
  0.55981445  0.4449768 ]
[Supersurface] frame 0: upsample_factor=2
[Coverage] frame 0: global_no_cov=0.0543, mi

In [ ]:
# =========================================================
# Save diagnostics
# =========================================================
if len(all_z_plane_stats) > 0:
    z_plane_stats_all = pd.concat(all_z_plane_stats, ignore_index=True)
    z_plane_stats_all.to_csv(
        os.path.join(diagnostic_dir, "z_plane_pred_stats.csv"),
        index=False,
    )
    print(f"Saved z_plane_stats: {len(z_plane_stats_all)} rows")

if len(all_coverage_stats) > 0:
    coverage_stats_all = pd.DataFrame(all_coverage_stats)
    coverage_stats_all.to_csv(
        os.path.join(diagnostic_dir, "coverage_stats.csv"),
        index=False,
    )
    print(f"Saved coverage_stats: {len(coverage_stats_all)} rows")

if fixed_target_z is not None:
    np.save(
        os.path.join(diagnostic_dir, "fixed_target_z.npy"),
        fixed_target_z.astype(np.float32),
    )
    print("Saved fixed_target_z.npy")

# Save processing order
np.save(
    os.path.join(diagnostic_dir, "processing_order.npy"),
    np.array(sorted(registered_mem_mapped.keys()), dtype=np.int32),
)

print("Diagnostics saved.")
print("Done.")